In [1]:
# Imports and environment first

from dotenv import load_dotenv
from sidekick_personal import Sidekick

load_dotenv(override=True)

True

## Step 3: the full Sidekick

The complete version lives in `sidekick.py` and `sidekick_tools.py`. The worker has the full toolkit: a headed browser and a sandbox filesystem through persistent MCP sessions, plus web search, Wikipedia, push notifications and `request_human_help`. The middleware stack from the start of this lab is all in place, and the evaluator loop checks each answer against your success criteria, using the worker's tool calls as evidence, sending the worker back to try again when needed.

This is the whole stack in one object. The worker is a Layer 3 create_agent, its tools are Layer 1 `@tool` functions alongside MCP servers, its memory is a Layer 2 checkpointer, and the loop around it is code you wrote yourself. You chose the altitude for each part.

Let us bring one to life and start with a quick browser errand. Watch the window open and drive itself. The top story on Hacker News changes through the day, so you will see a different headline than the one shown here.

In [2]:
sidekick = Sidekick()
await sidekick.setup()
print(f"Sidekick ready with {len(sidekick.tools)} tools")

Sidekick ready with 42 tools


## The finale: a real errand

Now a task worthy of the week: find me the best flight. This needs a plan, a stretch of real browsing, a file, and a push notification, and it ends with the Sidekick asking your permission. One detail to notice in `sidekick.py`: the worker's system prompt carries a practical trick, that Google Flights accepts a natural language query straight in the URL, giving your agent specialist abilities.

While it works, watch the browser window, and peek at the plan: the worker maintains a todo list through its `write_todos` tool, and the Sidekick exposes it as `sidekick.todos`.

In [3]:
flight_task = """Find me the best round-trip flight from fort lauderdale or Miami to San Andres Island, Colombia, leaving about a month from now
and returning a week later. I care about price first, then total journey time, and I would rather avoid
itineraries with two or more stops. Write your recommendation with the top three options to flights_personal.md,
then send me a push notification with the price of your top pick."""

flight_criteria = "flights_personal.md is written with three specific options including airline, times and price, plus a clear recommendation, and a push notification was sent with the recommended price."

history = await sidekick.run_turn(flight_task, flight_criteria, history=[])
print(history[-1]["content"])

Waiting for your approval:
Tool execution requires approval

Tool: request_human_help
Args: {'instructions': 'Please sign in to your Expedia account in the browser window. Click the "Sign in" button in the top-right corner and complete the login. Once you\'re signed in, I\'ll proceed with the flight searches.'}


The Sidekick has done the work, and now it is paused, asking permission to send the notification. Its plan tells the story:

In [4]:
for todo in sidekick.todos:
    print(f"[{todo['status']}] {todo['content']}")

[in_progress] Ask user to log into Expedia
[pending] Search flights on Expedia (FLL/MIA → ADZ, Sep 6-13)
[pending] Search flights on Google Flights (FLL/MIA → ADZ, Sep 6-13)
[pending] Compare results and pick top 3 options
[pending] Write recommendation to flights_personal.md
[pending] Send push notification with top pick price


We approve, the notification arrives on your phone, and the evaluator confirms the criteria were met, with the tool calls as its evidence.

In [5]:
if sidekick.paused:
    history = await sidekick.resume(history)
for entry in history[-2:]:
    print(f"[{entry['role']}] {entry['content']}\n")

[assistant] Model call limits exceeded: run limit (30/30)

[assistant] Evaluator: The assistant made extensive browser navigation attempts and ran code to extract flight data, but the final message is "Model call limits exceeded: run limit (30/30)." There is no evidence that flights_personal.md was ever written with three flight options, nor that a push notification was sent with the recommended price. The assistant also called request_human_help at one point, suggesting it got stuck. The task is incomplete — neither deliverable was produced.



## The Gradio app

`app.py` wraps all of this in a chat interface: a request box, a success criteria box, the Sidekick's plan updating live beside the chat while it works, and an Approve button that appears whenever it pauses for your say-so. Reset really does close the browser now, since the persistent MCP sessions shut down cleanly.

The look and feel lives in `styles.py` as a theme, a CSS constant and a JS constant. A Gradio 6 note: these are passed to `launch()`, not to `gr.Blocks()` as in older Gradio. Run the app from a terminal with `uv run app.py`, or launch the very same interface right here:

In [ ]:
from app import ui, LAUNCH_STYLE

app, local_url, share_url = ui.launch(**LAUNCH_STYLE)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Users/tonyhoward/Development/udemy/ed-donner-ai-engineer-agentic-track/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/tonyhoward/Development/udemy/ed-donner-ai-engineer-agentic-track/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/tonyhoward/Development/udemy/ed-donner-ai-engineer-agentic-track/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/tonyhoward/Development/udemy/ed-donner-

In [ ]:
app.close()

In [5]:
import gradio as gr
gr.close_all()

Closing server running on port: 7860


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations</h2>
            <span style="color:#00cc00;">You have built the Sidekick, and with it you have travelled the whole stack: the building blocks of Layer 1, the orchestration of LangGraph, the create_agent of Layer 3, the harness of Deep Agents, and now a real project that mixes them all, with middleware for planning, guardrails and human approval. That is a serious amount of capability, and you understand every layer of it. Wonderful work this week.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make the Sidekick truly yours. Start with a personal errand: try the Expedia version of the flight task, asking the Sidekick to sign in to your Expedia account first. When it reaches the login page it will call <code>request_human_help</code> and pause while you log in to its browser window, then press on with the search. Travel sites defend themselves against automation, so expect the occasional block wall; that is part of working with real-world agents. Then go further: gate the filesystem write tools behind the approval middleware in <code>sidekick.py</code>, add a tool of your own, and set the Sidekick a task you actually need done this week, with a clear success criterion.
            </span>
        </td>
    </tr>
</table>

## Exercise: personal errand - use Expedia (or travelocity) to find a flight instead of Google Flights, and use the browser to log in to a site

### Task

- figure out how to switch to expedia instead of google flights
- figure out how to log in to a site



input: Find me the best round-trip flight from fort lauderdale or Miami to San Andres Island, Colombia, leaving about a month from now
and returning a week later. I care about price first, then total journey time, and I would rather avoid
itineraries with two or more stops. Write your recommendation with the top three options to flights_personal.md,
then send me a push notification with the price of your top pick.

success criteria: Expedia and Google are used for the flight searches. flights_personal.md is written with three specific options including airline, times and price, plus a clear recommendation, and a push notification was sent with the recommended price.

In [1]:
# Imports and environment first
from dotenv import load_dotenv
from sidekick_personal import Sidekick
from app import ui, LAUNCH_STYLE

load_dotenv(override=True)

sidekick = Sidekick()
await sidekick.setup()

app, local_url, share_url = ui.launch(**LAUNCH_STYLE)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


/Users/tonyhoward/Development/udemy/ed-donner-ai-engineer-agentic-track/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/tonyhoward/Development/udemy/ed-donner-ai-engineer-agentic-track/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/tonyhoward/Development/udemy/ed-donner-ai-engineer-agentic-track/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/tonyhoward/Development/udemy/ed-donner-

In [2]:
import gradio as gr
gr.close_all()

Closing server running on port: 7861
